# 07 · The trap suite

**Goal:** check that the agent behaves well on five synthetic scenarios where the right answer is known:
clean confounding (proceed and recover the effect), no overlap (refuse), a collider offered as a control
(exclude it), a mediator offered as a control (exclude it), and a balance failure (re-specify, then refuse).

This notebook runs **offline**, so the 'agent' proposes the hidden reference answer: the counts test
the plumbing, not judgement. The real test uses Claude: `python -m causallab traps` (needs an API key).
Two runs per scenario here; the CLI uses ten.

## Setup

Imports and configuration. Every threshold and setting comes from `config.yaml`. Set `SYNTHETIC = True` to run on the synthetic file built by `python -m causallab prepare --synthetic`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from causallab.config import load_config, load_spec, results_dir

cfg = load_config()
SYNTHETIC = False   # True = run on the synthetic Expedia-like file (no Kaggle data needed)
pd.set_option("display.width", 140)

## The scenarios

**What:** each has product-analytics variable names and descriptions (what Claude sees), a hidden
reference answer and a true effect.

**Why the descriptions matter:** with Claude, the agent must infer from 'opened a support ticket after
the session' that the variable is a collider, without being told its role.

In [2]:
from causallab.synthetic import trap_scenarios
for sc in trap_scenarios(n=4000):
    print(f"{sc.name:18s} expected: {sc.expected:22s} true effect {sc.true_effect:.3f}  bad controls: {sc.bad_controls}")

clean_confounding  expected: proceed                true effect 0.100  bad controls: []
no_overlap         expected: refuse                 true effect 0.100  bad controls: []
collider           expected: proceed                true effect 0.099  bad controls: ['support_ticket_opened']
mediator           expected: proceed                true effect 0.134  bad controls: ['detail_page_seconds']
failing_balance    expected: respecify_then_refuse  true effect 0.106  bad controls: []


## Why the traps bite

**What:** AIPW with and without the bad control.

**Why:** adjusting for a collider opens a path between treatment and outcome; adjusting for a mediator
blocks part of the effect. Either way the estimate is wrong.

**Expect:** the collider pulls the estimate towards zero; the mediator roughly halves it.

In [3]:
from causallab import estimators as E
for sc in trap_scenarios(n=12000):
    if not sc.bad_controls:
        continue
    for label, cov in (("reference", sc.reference_adjustment), ("with bad control", sc.reference_adjustment + sc.bad_controls)):
        r = E.estimate_all(sc.df, sc.treatment, sc.outcome, cov, sc.group, cfg, which=("aipw",))["estimates"]["aipw"]
        print(f"{sc.name:9s} {label:17s} AIPW {r.estimate:.3f}  (truth {sc.true_effect:.3f})")

collider  reference         AIPW 0.088  (truth 0.100)


collider  with bad control  AIPW -0.019  (truth 0.100)


mediator  reference         AIPW 0.129  (truth 0.133)


mediator  with bad control  AIPW 0.054  (truth 0.133)


## Run the suite (offline, two runs per scenario)

**Expect:** a few minutes; every scenario 'as expected' in offline mode, since the stub knows the answer.
`fresh=True` gives these runs new ids, so nothing is reused from the CLI suite.

In [4]:
from causallab import trap_suite
res = trap_suite.run(offline=True, runs=2, cfg=cfg, fresh=True)
pd.DataFrame(res["summary"]).T

clean_confounding  seed 0: reported  expected=True


clean_confounding  seed 1: reported  expected=True


no_overlap         seed 0: refused   expected=True


no_overlap         seed 1: refused   expected=True


collider           seed 0: reported  expected=True


collider           seed 1: reported  expected=True


mediator           seed 0: reported  expected=True


mediator           seed 1: reported  expected=True


failing_balance    seed 0: refused   expected=True


failing_balance    seed 1: refused   expected=True


,expected,runs,as_expected,status_counts,bad_control_adjusted,respecified
clean_confounding,proceed and recover the effect,2,2,{'reported': 2},0,0
no_overlap,refuse,2,2,{'refused': 2},0,0
collider,exclude the collider and proceed,2,2,{'reported': 2},0,0
mediator,exclude the mediator and proceed,2,2,{'reported': 2},0,0
failing_balance,"re-specify, then refuse",2,2,{'refused': 2},0,2
